# Module 4.2 — CNN Building Blocks

**Module 4: CNNs, Pretrained Models & Deep Dream | Week 1 | Lecture 2**

---

Last session you saw convolution: a filter slides across an image and produces a feature map. Now let's build up the full vocabulary — pooling, stride, padding — and see how these blocks stack together into a CNN.

## Glossary

Today's terms describe the dials and pieces that turn convolution into a full CNN.

- **Padding** — zeros added around the edges of the input before convolution, so the filter can sit at edge positions. `padding=1` with a 3×3 filter keeps the output the same size as the input.
- **Stride** — how far the filter moves between each step. `stride=1` is one pixel at a time; `stride=2` skips every other pixel and halves the output size.
- **Channel** — one of the parallel feature maps a layer produces. A color image has 3 input channels (RGB); a conv layer with 64 filters produces 64 output channels.
- **Pooling** — a way to shrink a feature map by summarizing small regions. **Max pool** takes the largest value in each region; **average pool** takes the mean.
- **Downsample** — to reduce the spatial size of a feature map, usually via pooling or stride.
- **Receptive field** — the region of the original input that each output neuron "sees." Deeper layers have larger receptive fields, so they can detect bigger patterns.
- **Invariance** — robustness to a transformation. Pooling adds a small amount of translation invariance: shifting the input by one pixel barely changes the pooled output.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

---

## nn.Conv2d: PyTorch's Convolutional Layer

In PyTorch, a convolutional layer is `nn.Conv2d`. The key arguments:

- `in_channels` — how many channels the input has (1 for grayscale, 3 for RGB)
- `out_channels` — how many filters (each produces one feature map)
- `kernel_size` — the filter size (3 means 3×3)

Each filter learns its own pattern. 16 filters = 16 different feature maps.

In [ ]:
# Create a conv layer and see what it does to a batch of images
conv = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)

# Fake batch: 4 grayscale 28×28 images
x = torch.randn(4, 1, 28, 28)
out = conv(x)

print(f"Input shape:  {x.shape}")    # [4, 1, 28, 28]
print(f"Output shape: {out.shape}")  # [4, 16, 28, 28]
print(f"\n16 filters × (3×3 kernel × 1 input channel + 1 bias) = {conv.weight.numel() + conv.bias.numel()} parameters")

The output has 16 channels — one feature map per filter. Each feature map is a "view" of the image through a different lens.

---

## Padding

Without padding, the output shrinks. A 3×3 filter on a 28×28 image produces a 26×26 output (lost 1 pixel on each side).

**Padding** adds zeros around the border so the output stays the same size as the input.

- `padding=0` — no padding (output shrinks)
- `padding=1` — one pixel of zeros on each side (output same size for 3×3 kernel)
- Rule of thumb: `padding = kernel_size // 2` keeps the size the same

In [ ]:
x = torch.randn(1, 1, 8, 8)

for pad in [0, 1, 2]:
    conv = nn.Conv2d(1, 1, kernel_size=3, padding=pad)
    out = conv(x)
    print(f"padding={pad}:  {x.shape[-1]}×{x.shape[-1]} → {out.shape[-1]}×{out.shape[-1]}")

---

## Stride

By default, the filter moves one pixel at a time (stride=1). With **stride=2**, it skips every other position — cutting the output size in half.

Stride is another way to downsample (shrink) the feature map.

In [ ]:
x = torch.randn(1, 1, 8, 8)

for stride in [1, 2]:
    conv = nn.Conv2d(1, 1, kernel_size=3, padding=1, stride=stride)
    out = conv(x)
    print(f"stride={stride}:  {x.shape[-1]}×{x.shape[-1]} → {out.shape[-1]}×{out.shape[-1]}")

---

## Pooling

**Pooling** shrinks the feature map by summarizing small regions. The most common type is **max pooling**: take the maximum value in each 2×2 block.

Why pool?
1. **Reduces size** — less computation in later layers
2. **Adds invariance** — small shifts in the input don't change the pooled output

In [ ]:
# Max pooling with real numbers
feature_map = torch.tensor([
    [1.0, 0.5, 2.0, 0.1],
    [0.3, 1.8, 0.7, 0.4],
    [0.9, 0.2, 3.0, 1.5],
    [0.6, 0.8, 0.3, 2.1],
]).unsqueeze(0).unsqueeze(0)  # shape: [1, 1, 4, 4]

pool = nn.MaxPool2d(kernel_size=2)
pooled = pool(feature_map)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Input
data = feature_map.squeeze().numpy()
axes[0].imshow(data, cmap='YlOrRd', vmin=0, vmax=3)
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f"{data[i,j]:.1f}", ha='center', va='center',
                     fontsize=13, fontweight='bold')
# Draw 2×2 blocks
for r in [0, 2]:
    for c in [0, 2]:
        rect = plt.Rectangle((c-0.5, r-0.5), 2, 2, edgecolor='#333',
                              linewidth=2, facecolor='none')
        axes[0].add_patch(rect)
axes[0].set_title('Input (4×4)', fontweight='bold')
axes[0].set_xticks([])
axes[0].set_yticks([])

# Pooled output
data_p = pooled.squeeze().numpy()
axes[1].imshow(data_p, cmap='YlOrRd', vmin=0, vmax=3)
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{data_p[i,j]:.1f}", ha='center', va='center',
                     fontsize=15, fontweight='bold')
axes[1].set_title('After MaxPool2d(2) → (2×2)', fontweight='bold')
axes[1].set_xticks([])
axes[1].set_yticks([])

plt.suptitle('Max Pooling: Keep the Strongest Activation in Each Block',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

Each 2×2 block collapses to its maximum value. The 4×4 feature map becomes 2×2. The strongest signals survive.

---

## The Full CNN Pipeline

A typical CNN stacks these blocks in a repeating pattern:

**Conv → ReLU → Pool → Conv → ReLU → Pool → ... → Flatten → FC → Output**

Each Conv+Pool block extracts features at a finer level and shrinks the spatial dimensions. At the end, we flatten and use fully-connected layers (like Module 3) for classification.

In [ ]:
# Pipeline diagram
fig, ax = plt.subplots(figsize=(16, 4))
ax.set_xlim(0, 16)
ax.set_ylim(0, 4)
ax.axis('off')

blocks = [
    (1.0,  'Input\n1×28×28',     '#999',    1.6),
    (3.2,  'Conv1\n16×28×28',    '#4A90D9', 1.6),
    (5.0,  'ReLU\n+ Pool',       '#F0AD4E', 1.3),
    (6.7,  'Conv2\n32×14×14',    '#4A90D9', 1.6),
    (8.5,  'ReLU\n+ Pool',       '#F0AD4E', 1.3),
    (10.3, 'Flatten\n32×7×7\n= 1568',  '#999', 1.6),
    (12.3, 'FC\n128',            '#5CB85C', 1.3),
    (14.0, 'Output\n10',         '#D9534F', 1.3),
]

for x, label, color, w in blocks:
    box = mpatches.FancyBboxPatch(
        (x - w/2, 1.0), w, 1.8, boxstyle='round,pad=0.12',
        facecolor=color, edgecolor='white', linewidth=2, alpha=0.9
    )
    ax.add_patch(box)
    ax.text(x, 1.9, label, ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')

# Arrows
arrow_xs = [(1.8, 2.4), (4.0, 4.35), (5.65, 5.9), (7.5, 7.85),
            (9.15, 9.5), (11.1, 11.65), (12.95, 13.35)]
for start, end in arrow_xs:
    ax.annotate('', xy=(end, 1.9), xytext=(start, 1.9),
                arrowprops=dict(arrowstyle='-|>', color='#333', lw=1.5))

fig.suptitle('A Typical CNN Architecture', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Dimension Tracking

One of the trickiest parts of CNNs is tracking how dimensions change through each layer. Let's trace a 28×28 MNIST image through a simple CNN.

In [ ]:
# Trace dimensions through a CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)   # 1→16 channels
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # 16→32 channels
        self.pool = nn.MaxPool2d(2)                                # halves spatial dims
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        print(f"  Input:        {x.shape}")
        x = self.pool(torch.relu(self.conv1(x)))
        print(f"  Conv1+Pool:   {x.shape}")
        x = self.pool(torch.relu(self.conv2(x)))
        print(f"  Conv2+Pool:   {x.shape}")
        x = x.flatten(1)
        print(f"  Flatten:      {x.shape}")
        x = torch.relu(self.fc1(x))
        print(f"  FC1:          {x.shape}")
        x = self.fc2(x)
        print(f"  FC2 (output): {x.shape}")
        return x

model = SimpleCNN()
dummy = torch.randn(1, 1, 28, 28)
print("Dimension trace:")
_ = model(dummy)

The pattern:
- **Channels increase:** 1 → 16 → 32 (more features detected)
- **Spatial dimensions decrease:** 28 → 14 → 7 (max pool halves each time)
- **Flatten** before fully-connected: 32 × 7 × 7 = 1,568

The output formula for conv/pool:

$$\text{output\_size} = \frac{\text{input\_size} - \text{kernel\_size} + 2 \times \text{padding}}{\text{stride}} + 1$$

For our setup (kernel=3, padding=1, stride=1): output = input. Then MaxPool2d(2) halves it.

---

## Feature Hierarchies

As you go deeper in a CNN, the features get more abstract:

- **Layer 1:** Edges, lines, simple textures
- **Layer 2:** Corners, curves, combinations of edges
- **Layer 3+:** Parts of objects (eyes, wheels, petals)
- **Deep layers:** Whole objects or concepts

This hierarchy is what makes CNNs so powerful. Each layer builds on the patterns found by the previous one.

(For now this is a claim — we'll *see* it empirically in 4.6, where Deep Dream visualizes what each layer has learned.)

In [ ]:
# Feature hierarchy diagram
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 3.5)
ax.axis('off')

layers = [
    (2,   'Layer 1',    '#4A90D9', 'Edges\nLines'),
    (5,   'Layer 2',    '#5CB85C', 'Corners\nTextures'),
    (8,   'Layer 3',    '#F0AD4E', 'Parts\nShapes'),
    (11,  'Deep Layers', '#D9534F', 'Objects\nConcepts'),
]

for x, title, color, examples in layers:
    box = mpatches.FancyBboxPatch(
        (x - 1.2, 0.8), 2.4, 1.8, boxstyle='round,pad=0.15',
        facecolor=color, edgecolor='white', linewidth=2, alpha=0.9
    )
    ax.add_patch(box)
    ax.text(x, 2.0, title, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')
    ax.text(x, 1.25, examples, ha='center', va='center',
            fontsize=9, color='white', alpha=0.9)
    ax.text(x, 0.3, '→ more abstract →', ha='center',
            fontsize=8, color='#666', fontstyle='italic')

for start, end in [(3.2, 3.8), (6.2, 6.8), (9.2, 9.8)]:
    ax.annotate('', xy=(end, 1.7), xytext=(start, 1.7),
                arrowprops=dict(arrowstyle='-|>', color='#333', lw=2))

fig.suptitle('Feature Hierarchy: Simple → Complex',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Parameter Count: MLP vs CNN

Let's compare parameter counts for classifying 28×28 grayscale images into 10 classes.

In [ ]:
# MLP from Module 3
mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 10),
)

# CNN
cnn = SimpleCNN()

mlp_params = sum(p.numel() for p in mlp.parameters())
cnn_params = sum(p.numel() for p in cnn.parameters())

print(f"MLP parameters: {mlp_params:>10,}")
print(f"CNN parameters: {cnn_params:>10,}")
print(f"\nThe CNN has {'more' if cnn_params > mlp_params else 'fewer'} parameters ")
print(f"but much better spatial awareness and translation invariance.")

# Breakdown
print("\nCNN layer-by-layer:")
for name, param in cnn.named_parameters():
    print(f"  {name:15s}  {str(list(param.shape)):20s}  {param.numel():>8,}")

Notice: the conv layers have very few parameters (hundreds), but the first FC layer (`fc1`) has the most — that's the cost of going from 2D feature maps to 1D classification. This is why architectures try to reduce the spatial dimensions as much as possible before flattening.

---

## Exercises

### Exercise 1: Dimension math

An input image is 32×32 with 3 channels (RGB). Trace the dimensions through:
1. `Conv2d(3, 32, kernel_size=3, padding=1)` → shape?
2. `MaxPool2d(2)` → shape?
3. `Conv2d(32, 64, kernel_size=3, padding=1)` → shape?
4. `MaxPool2d(2)` → shape?
5. `Flatten` → shape?

### Exercise 2: Parameter counting

How many parameters (weights + biases) does `nn.Conv2d(3, 16, kernel_size=5)` have? Show your work.

### Exercise 3: Build it in code

Create a `nn.Sequential` model that implements the architecture from Exercise 1, ending with a `nn.Linear` layer for 10-class output. Verify the dimensions by passing a random tensor through it.

In [ ]:
# Exercises: Your code here


---

## Summary

**CNN building blocks:**
- `nn.Conv2d` — slides learned filters across the image, produces feature maps
- `nn.MaxPool2d` — shrinks feature maps by keeping the max in each block
- **Padding** — adds zeros around the border to control output size
- **Stride** — controls how far the filter moves at each step

**The CNN pipeline:** Conv → ReLU → Pool (repeat) → Flatten → FC → Output

**Feature hierarchy:** early layers detect edges, deeper layers detect increasingly complex patterns.

**Dimension tracking:** channels increase (more features), spatial dimensions decrease (pooling).

---

**Next session:** A brief history of CNN architectures — from LeNet to ResNet.